# 04 - Fine-tune on YOUR OWN frames

Loads the LettuceMOTS-trained base weights and continues training at a **low learning rate** on your own cultivator-camera frames (supplied later). Keeps a single class.

**Your frames must be labeled in YOLO detection format** (images + a mirrored `labels/` tree, or a data yaml). Point `OWN_DATA_YAML` at a single-class yaml. Use only real captured/annotated frames - no synthetic or AI-generated images.

> **DO NOT RUN** until your frames + labels exist and the config below points at them. This notebook ships **unrun** on purpose.

In [ ]:
import os, sys
from pathlib import Path
REPO_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
                 if (p / "croprow" / "utils.py").is_file())
sys.path.insert(0, str(REPO_ROOT))
from croprow import utils as U
CW = REPO_ROOT / "croprow"
DATA_DIR = CW / "data"
MODELS_DIR = CW / "models"
RUNS_DIR = CW / "runs"
RESULTS_MD = CW / "RESULTS.md"

# ===================== CONFIG (edit here only) =====================
# Base weights from 03_train (LettuceMOTS). Fall back to models/best.pt.
BASE_WEIGHTS = str(MODELS_DIR / "best.pt")

# Path to YOUR single-class data yaml (train/val over your own frames).
# Prefer the env var; else edit the default. This path is intentionally a
# placeholder -- fill it in when your frames are ready.
OWN_DATA_YAML = os.environ.get("OWN_DATA_YAML", r"D:\path\to\your_frames\own.yaml")

IMGSZ      = 640
EPOCHS     = 50
BATCH      = 16
LR0        = 0.001                 # low LR for fine-tuning (10x below 03)
LRF        = 0.01
OPTIMIZER  = "auto"
FREEZE     = 10                    # freeze backbone layers (0 to disable)
PATIENCE   = 20
WORKERS    = 8
DEVICE     = 0
SEED       = 42

RUN_NAME   = f"finetune_own_{IMGSZ}"
# ===================================================================
print("base weights:", BASE_WEIGHTS)
print("own data yaml:", OWN_DATA_YAML)

## Environment check

In [ ]:
# This notebook needs the training/inference stack (torch + ultralytics),
# NOT installed in the light 01/02 env. Install into a Python 3.11 venv with
# numpy<2 -- see croprow/requirements-train.txt and croprow/README.md.
try:
    import torch
    from ultralytics import YOLO
    import ultralytics
    print("torch      :", torch.__version__)
    print("ultralytics:", ultralytics.__version__)
    print("CUDA avail :", torch.cuda.is_available(),
          "|", (torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only"))
except ModuleNotFoundError as e:
    raise ModuleNotFoundError(
        f"Missing training dependency: {e.name}. Install croprow/requirements-train.txt "
        "into a Python 3.11 (numpy<2) venv before running this notebook."
    ) from e

## Guard: confirm inputs exist before training

Stops with a clear message rather than fabricating data if the base weights or your data yaml are missing.

In [ ]:
if not Path(BASE_WEIGHTS).is_file():
    raise FileNotFoundError(
        f"Base weights not found: {BASE_WEIGHTS}. Run 03_train first, or set "
        "BASE_WEIGHTS to your trained checkpoint.")
if not Path(OWN_DATA_YAML).is_file():
    raise FileNotFoundError(
        f"Own-frames data yaml not found: {OWN_DATA_YAML}. Provide your labeled "
        "frames (single class) and point OWN_DATA_YAML at their yaml.")
print("inputs OK")

## Fine-tune

In [ ]:
model = YOLO(BASE_WEIGHTS)
results = model.train(
    data=OWN_DATA_YAML,
    imgsz=IMGSZ,
    epochs=EPOCHS,
    batch=BATCH,
    lr0=LR0,
    lrf=LRF,
    optimizer=OPTIMIZER,
    freeze=FREEZE,
    patience=PATIENCE,
    workers=WORKERS,
    device=DEVICE,
    seed=SEED,
    project=str(RUNS_DIR),
    name=RUN_NAME,
    exist_ok=True,
)
save_dir = Path(model.trainer.save_dir)
best = save_dir / "weights" / "best.pt"
print("fine-tuned best:", best)

import shutil
if best.is_file():
    dst = MODELS_DIR / "best_finetuned.pt"
    shutil.copy2(best, dst)
    print("copied ->", dst)

Evaluate the fine-tuned model on your own val split in `05_evaluate` (reported separately from LettuceMOTS - never merged).